# 04 — Disease Modules & Network Overlap *(optional)*

**Network Medicine Workshop · Kidney Disease · Optional Part 4**

This notebook is an extension, not a prerequisite for the core story — run it if there's time. It has two parts:

**Part A — Is your module really a module?** We come back to the question we skipped in Notebook 2: is your DE nodes' connectivity in each network more than you'd expect from a random set of the same size and degree distribution? (Same degree-preserving permutation test used implicitly elsewhere, made explicit here with a z-score and p-value.)

**Part B — How does your disease relate to other diseases, topologically?** This is the classic network medicine "disease module overlap" idea (Menche et al., *Science* 2015): two diseases whose gene modules sit close together in the interactome tend to share mechanisms, comorbidities, and sometimes treatments — even if the gene lists themselves barely overlap. We compare your kidney-disease module against:
- **C3 glomerulopathy** — a kidney disease with a distinct but related mechanism (complement dysregulation), expected to show network *proximity*
- **An unrelated disease** (default: Parkinson's disease, change if you'd like) — expected to show network *separation*

The statistic is the **network separation S_AB**:

```
S_AB = <d_AB> - (<d_AA> + <d_BB>) / 2
```

`d_AA` / `d_BB` = average shortest-path distance from each node in a module to its nearest neighbor within the *same* module. `d_AB` = average shortest-path distance between the two modules. **S_AB < 0 means the modules overlap/sit close together** (shared mechanism); **S_AB > 0 means they're topologically separated** (distinct mechanisms).

**Timing: ~20 min.** Requires Notebooks 1–2 (and ideally 3, for the richest version of "your module").


## Setup

In [ ]:
!pip install -q scipy networkx requests mygene matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pickle, time
import pandas as pd
import numpy as np
import networkx as nx
import requests

BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"
PROC_DIR = os.path.join(BASE_DIR, "processed")

with open(os.path.join(PROC_DIR, "modules.pkl"), "rb") as f:
    modules = pickle.load(f)

graphs = {}
for layer in ["ppi", "transcriptome", "metabolite"]:
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "rb") as f:
        graphs[layer] = pickle.load(f)

print("Loaded modules for layers:", list(modules.keys()))


---
# Part A — Module significance

For each network layer, we test whether the DE-mapped nodes are more connected (bigger largest connected component) than a random node set drawn from the same **degree distribution** — controlling for the fact that hub nodes are just generally more "connected" regardless of biology.


In [ ]:
def degree_bins(G, n_bins=10):
    degrees = dict(G.degree())
    deg_values = np.array(list(degrees.values()))
    edges = np.unique(np.quantile(deg_values, np.linspace(0, 1, n_bins + 1)))
    bin_of_node, nodes_in_bin = {}, {i: [] for i in range(len(edges) - 1)}
    for node, d in degrees.items():
        b = min(np.searchsorted(edges, d, side="right") - 1, len(edges) - 2)
        bin_of_node[node] = b
        nodes_in_bin[b].append(node)
    return bin_of_node, nodes_in_bin

def largest_cc_size(G, nodes):
    if len(nodes) < 2:
        return len(nodes)
    sub = G.subgraph(nodes)
    if sub.number_of_edges() == 0:
        return 1
    return len(max(nx.connected_components(sub), key=len))

def module_significance(G, seed_nodes, n_perm=1000, seed=0):
    rng = np.random.default_rng(seed)
    bin_of_node, nodes_in_bin = degree_bins(G)
    observed = largest_cc_size(G, seed_nodes)

    bin_counts = {}
    for n in seed_nodes:
        b = bin_of_node[n]
        bin_counts[b] = bin_counts.get(b, 0) + 1

    null_dist = []
    for _ in range(n_perm):
        random_set = []
        for b, count in bin_counts.items():
            pool = nodes_in_bin[b]
            random_set.extend(rng.choice(pool, size=min(count, len(pool)), replace=False))
        null_dist.append(largest_cc_size(G, random_set))

    null_dist = np.array(null_dist)
    z = (observed - null_dist.mean()) / (null_dist.std() + 1e-9)
    p = (null_dist >= observed).mean()
    return {"observed_lcc": observed, "null_mean": null_dist.mean(),
            "null_std": null_dist.std(), "z_score": z, "p_value": p, "null_dist": null_dist}

connectivity_results = {}
for layer, G in graphs.items():
    seed_nodes = modules[layer]["seed_nodes"]
    if len(seed_nodes) < 5:
        print(f"[{layer}] too few mapped nodes ({len(seed_nodes)}) to test — skipping")
        continue
    res = module_significance(G, seed_nodes, n_perm=1000)
    connectivity_results[layer] = res
    print(f"[{layer}] observed LCC={res['observed_lcc']} vs random {res['null_mean']:.1f} ± {res['null_std']:.1f}  "
          f"→ z={res['z_score']:.2f}, p={res['p_value']:.4f}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(connectivity_results), figsize=(5*len(connectivity_results), 4))
if len(connectivity_results) == 1:
    axes = [axes]

for ax, (layer, res) in zip(axes, connectivity_results.items()):
    ax.hist(res["null_dist"], bins=30, color="lightgray", edgecolor="white")
    ax.axvline(res["observed_lcc"], color="crimson", linewidth=2, label="observed")
    ax.set_title(f"{layer}\nz={res['z_score']:.2f}, p={res['p_value']:.4f}")
    ax.set_xlabel("largest connected component size")
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, "connectivity_test.png"), dpi=150)
plt.show()


**Reading this with the group:** a large positive z-score (small p-value) says these genes/proteins/metabolites cluster together in the network far more than chance — evidence of a real, shared underlying process. A z-score near zero says the DE list, at least as mapped onto this network, doesn't behave like a coherent module — a legitimate (if less exciting) result worth discussing openly.


---
# Part B — Disease-disease network separation (S_AB)

## Step 1 — Choose "your" module

We use the richest available representation of your kidney-disease signal: the **cross-omics bridge module** from Notebook 3 if you've run it (protein module + metabolite-linked genes + connecting nodes), falling back to the plain PPI protein module otherwise.


In [ ]:
bridge_path = os.path.join(PROC_DIR, "bridge_module.pkl")
if os.path.exists(bridge_path):
    with open(bridge_path, "rb") as f:
        MODULE_A = pickle.load(f)
    module_a_source = "bridge module (Notebook 3)"
else:
    MODULE_A = modules["ppi"]["seed_nodes"]
    module_a_source = "PPI protein module (Notebook 2)"

G = graphs["ppi"]
MODULE_A = MODULE_A & set(G.nodes())
print(f"Module A = {module_a_source}: {len(MODULE_A)} nodes in the PPI network")


## Step 2 — Fetch reference disease gene sets from Open Targets

Same public API as Notebook 3: search for a disease name, get its EFO ID, pull its associated target genes.


In [ ]:
OT_API = "https://api.platform.opentargets.org/api/v4/graphql"

SEARCH_QUERY = """
query searchDisease($q: String!) {
  search(queryString: $q, entityNames: ["disease"], page: {index: 0, size: 5}) {
    hits { id name entity }
  }
}
"""

ASSOC_QUERY = """
query diseaseTargets($efoId: String!, $size: Int!) {
  disease(efoId: $efoId) {
    associatedTargets(page: {index: 0, size: $size}) {
      rows { target { id approvedSymbol } score }
    }
  }
}
"""

def fetch_disease_genes(disease_name, top_n=300, min_score=0.0):
    r = requests.post(OT_API, json={"query": SEARCH_QUERY, "variables": {"q": disease_name}}, timeout=30)
    r.raise_for_status()
    hits = r.json()["data"]["search"]["hits"]
    if not hits:
        print(f"[warn] no disease match for '{disease_name}'")
        return None, set()
    efo_id = hits[0]["id"]
    print(f"'{disease_name}' -> {efo_id} ({hits[0]['name']})")

    r = requests.post(OT_API, json={"query": ASSOC_QUERY,
                                      "variables": {"efoId": efo_id, "size": top_n}}, timeout=30)
    r.raise_for_status()
    rows = r.json()["data"]["disease"]["associatedTargets"]["rows"]
    genes = {row["target"]["approvedSymbol"] for row in rows if row["score"] >= min_score}
    print(f"  -> {len(genes)} associated genes (score >= {min_score})")
    return efo_id, genes

DISEASE_SIMILAR = "C3 glomerulopathy"     # expected to be network-close to your kidney data
DISEASE_DIFFERENT = "Parkinson disease"   # expected to be network-far; change to whatever "unrelated" means for your audience

_, similar_gene_symbols = fetch_disease_genes(DISEASE_SIMILAR)
_, different_gene_symbols = fetch_disease_genes(DISEASE_DIFFERENT)


## Step 3 — Map disease genes to NCBI Gene IDs and restrict to the PPI network

Same batch-mapping approach as Notebook 1.


In [ ]:
import mygene
mg = mygene.MyGeneInfo()

def symbols_to_ncbi_in_network(symbols, G):
    result = mg.querymany(list(symbols), scopes="symbol,alias", fields="entrezgene",
                           species="human", returnall=True)
    ids = set()
    for hit in result["out"]:
        if "entrezgene" in hit:
            ids.add(str(hit["entrezgene"]))
    return ids & set(G.nodes())

MODULE_B = symbols_to_ncbi_in_network(similar_gene_symbols, G)     # C3 glomerulopathy
MODULE_C = symbols_to_ncbi_in_network(different_gene_symbols, G)   # unrelated disease

print(f"Module B ({DISEASE_SIMILAR}): {len(MODULE_B)} nodes in PPI network")
print(f"Module C ({DISEASE_DIFFERENT}): {len(MODULE_C)} nodes in PPI network")
print(f"Direct gene overlap A∩B: {len(MODULE_A & MODULE_B)}   A∩C: {len(MODULE_A & MODULE_C)}")


## Step 4 — Compute network separation S_AB

We need, for two modules A and B: the average within-module distance for each (`d_AA`, `d_BB`) and the average between-module distance (`d_AB`). All of this comes from **two multi-source Dijkstra calls** (one seeded from A, one from B) rather than any all-pairs computation — this is what keeps it tractable at network sizes up to ~20k nodes.


In [ ]:
from scipy.sparse.csgraph import dijkstra

nodes = list(G.nodes())
node_idx = {n: i for i, n in enumerate(nodes)}
Adj = nx.to_scipy_sparse_array(G, nodelist=nodes, weight=None, format="csr")

def multi_source_distances(module):
    idx = [node_idx[n] for n in module if n in node_idx]
    return dijkstra(csgraph=Adj, directed=False, indices=idx, unweighted=True), idx

def network_separation(G, module_a, module_b, label_a="A", label_b="B"):
    dist_from_a, idx_a = multi_source_distances(module_a)   # shape (|A|, n_nodes)
    dist_from_b, idx_b = multi_source_distances(module_b)

    # d_AA: for each node in A, min distance to another node in A (exclude self), averaged
    sub_aa = dist_from_a[:, idx_a].copy()
    np.fill_diagonal(sub_aa, np.inf)
    d_aa = np.nanmean(np.min(sub_aa, axis=1)[np.isfinite(np.min(sub_aa, axis=1))])

    sub_bb = dist_from_b[:, idx_b].copy()
    np.fill_diagonal(sub_bb, np.inf)
    d_bb = np.nanmean(np.min(sub_bb, axis=1)[np.isfinite(np.min(sub_bb, axis=1))])

    # d_AB: distance from each A node to nearest B node, AND each B node to nearest A node, pooled
    sub_ab = dist_from_a[:, idx_b]   # rows=A, cols=B
    d_a_to_b = np.min(sub_ab, axis=1)
    d_b_to_a = np.min(sub_ab, axis=0)   # symmetric graph, so this equals "each B node's min distance to A"
    pooled = np.concatenate([d_a_to_b, d_b_to_a])
    d_ab = np.nanmean(pooled[np.isfinite(pooled)])

    s_ab = d_ab - (d_aa + d_bb) / 2
    print(f"[{label_a} vs {label_b}]  d_AA={d_aa:.2f}  d_BB={d_bb:.2f}  d_AB={d_ab:.2f}  ->  S_AB = {s_ab:.3f}")
    return {"d_AA": d_aa, "d_BB": d_bb, "d_AB": d_ab, "S_AB": s_ab}

sep_similar = network_separation(G, MODULE_A, MODULE_B, "your module", DISEASE_SIMILAR)
sep_different = network_separation(G, MODULE_A, MODULE_C, "your module", DISEASE_DIFFERENT)


**Interpretation:** `S_AB < 0` → the two modules overlap / sit close together in the network → shared mechanism. `S_AB > 0` → the modules are topologically separated → distinct mechanisms. If the biology behaves the way we'd expect, `S_AB(your module, C3 glomerulopathy)` should be noticeably smaller (more negative / less positive) than `S_AB(your module, {unrelated disease})` — your kidney-disease signal sits closer, in network space, to a related kidney disease than to an unrelated one.

This is also a nice moment to contrast with the **direct gene overlap** numbers from Step 3: it's common for two related diseases to show striking network proximity (small/negative S_AB) even when they share almost no genes directly — that's the whole point of doing this in network space rather than just intersecting lists.


In [ ]:
labels = [DISEASE_SIMILAR, DISEASE_DIFFERENT]
values = [sep_similar["S_AB"], sep_different["S_AB"]]

plt.figure(figsize=(5, 4))
colors = ["#55A868" if v < 0 else "#C44E52" for v in values]
plt.bar(labels, values, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Network separation S$_{AB}$")
plt.title("Your module vs. reference diseases")
plt.xticks(rotation=15)
for i, v in enumerate(values):
    plt.text(i, v + (0.02 if v >= 0 else -0.06), f"{v:.2f}", ha="center")
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, "s_ab_comparison.png"), dpi=150)
plt.show()


---
## Discussion prompts for the group

- Did S_AB come out in the expected direction (closer to the related disease, farther from the unrelated one)? If not, is that a data-quality issue, or a genuinely interesting result?
- How does the network-based proximity compare to the direct gene overlap numbers from Step 3? Which one would you trust more, and why?
- If you had a third or fourth candidate "similar" disease, how would you decide which one is *most* related — just compare S_AB across several?
- This is exactly the kind of analysis behind disease-network maps used for drug repurposing (a drug's target module close to your disease module is a repurposing candidate) — is that a direction worth pursuing with your bridge-module genes from Notebook 3?
